# Third-Party Breach News Classification — SLM Fine-Tuning Benchmark

**Proje:** Kucuk dil modellerini (SmolLM2-360M, TinyLlama-1.1B, Qwen2.5-1.5B, Gemma 4 E2B) siber guvenlik haberlerini **third-party breach / not breach** olarak siniflandiracak sekilde fine-tune etmek ve karsilastirmak.

> **Adimlar:** Kurulum → Veri → Zero-Shot Baseline → Fine-Tuning → Degerlendirme → Karsilastirma

---
*Runtime: GPU (T4 veya daha iyi onerilen). Runtime → Change runtime type → T4 GPU*

## 1. Paket Kurulumu

In [ ]:
# Gerekli paketleri kur
!pip install -q transformers==4.44.0 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.43.3
!pip install -q scikit-learn matplotlib datasets

import importlib, sys
print("Kurulum tamamlandi.")
print(f"Python: {sys.version.split()[0]}")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Google Drive Baglantisi & Dosyalari Yukle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# GitHub reposunu clonla VEYA Drive'dan kopyala
# Secim 1: GitHub
REPO_URL = "https://github.com/KULLANICI_ADI/third-party-breach-classification"
# !git clone {REPO_URL} /content/project

# Secim 2: Drive'dan kopyala (Drive'a projeyi yuklemeniz gerekir)
DRIVE_PATH = "/content/drive/MyDrive/third_party_breach_project"
if os.path.exists(DRIVE_PATH):
    shutil.copytree(DRIVE_PATH, "/content/project", dirs_exist_ok=True)
    print("Drive'dan kopyalandi.")
else:
    os.makedirs("/content/project/data", exist_ok=True)
    print("Proje dizini olusturuldu. Dosyalari manuel yukleyin.")

%cd /content/project
!ls -la

## 3. Veri Hazirlama

In [ ]:
# JSON dosyalarinin varlığını kontrol et
import os
assert os.path.exists("third_party_news.json"), "third_party_news.json bulunamadi!"
assert os.path.exists("not_third_party_news.json"), "not_third_party_news.json bulunamadi!"
print("Dosyalar mevcut.")

In [ ]:
# Veri hazirlama scriptini calistir
!python data_prep.py

In [ ]:
# Veriyi gorsellesdir
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nSinif dagilimi (Train):")
print(train_df["label"].value_counts().rename({0: "Not Breach", 1: "Breach"}))

# Metin uzunlugu dagilimi
train_df["char_len"] = train_df["text"].str.len()
train_df["char_len"].hist(bins=40, figsize=(10, 4), color="steelblue", edgecolor="white")
plt.title("Metin Uzunlugu Dagilimi (Karakter)")
plt.xlabel("Karakter sayisi")
plt.ylabel("Frekans")
plt.axvline(train_df["char_len"].mean(), color="red", linestyle="--", label=f"Ort: {train_df['char_len'].mean():.0f}")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/text_length_dist.png", dpi=120)
plt.show()
print("Grafik kaydedildi.")

## 4. Model Konfigurasyonu

In [ ]:
# Kullanilacak modeller
MODELS = {
    "smollm2"  : "HuggingFaceTB/SmolLM2-360M",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen"     : "Qwen/Qwen2.5-1.5B",
    "gemma"    : "google/gemma-4-e2b-it",   # Hugging Face erisimi gerektirir
}

# Bu deneyde kullanilacak modeller (en az 3 secin)
SELECTED_MODELS = ["smollm2", "tinyllama", "qwen"]  # Gemma icin "gemma" ekle

QUANTIZE_MODELS = {"qwen", "gemma"}  # Bu modeller QLoRA (4-bit) kullanacak

print("Secili modeller:")
for k in SELECTED_MODELS:
    q = "(QLoRA)" if k in QUANTIZE_MODELS else "(LoRA)"
    print(f"  {k:12s} {q} -> {MODELS[k]}")

## 5. Zero-Shot Baseline

In [ ]:
# Zero-shot: Fine-tuning oncesi temel performans
# Sadece kucuk/hizli modelde yapilir (tinyllama ornegi)

from transformers import pipeline
import csv, torch

ZERO_SHOT_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Zero-shot modeli yukleniyor: {ZERO_SHOT_MODEL}")
gen = pipeline(
    "text-generation",
    model=ZERO_SHOT_MODEL,
    device_map="auto",
    max_new_tokens=5,
    torch_dtype=torch.float16,
)

PROMPT = (
    "Is the following news about a third-party data breach caused by a vendor, "
    "supplier, or partner? Answer yes or no only.\n\nArticle:\n{text}\n\nAnswer:"
)

def load_csv_texts(path):
    texts, labels = [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(int(row["label"]))
    return texts, labels

# Test setinin ilk 50 orneginde dene (hiz icin)
texts, labels = load_csv_texts("data/test.csv")
texts_sample, labels_sample = texts[:50], labels[:50]

preds = []
for text in texts_sample:
    prompt = PROMPT.format(text=text[:600])
    out    = gen(prompt)[0]["generated_text"]
    answer = out.split("Answer:")[-1].strip().lower()
    preds.append(1 if answer.startswith("yes") else 0)

from sklearn.metrics import accuracy_score, f1_score
acc = accuracy_score(labels_sample, preds)
f1  = f1_score(labels_sample, preds, average="macro")
print(f"\nZero-Shot Sonuclari (n=50):")
print(f"  Accuracy : {acc:.4f}")
print(f"  F1 Macro : {f1:.4f}")

zero_shot_results = {"accuracy": acc, "f1": f1, "model": ZERO_SHOT_MODEL}

## 6. Fine-Tuning (Her Model Icin)

In [ ]:
import subprocess, json, os

os.makedirs("outputs", exist_ok=True)
all_results = {}

for model_key in SELECTED_MODELS:
    quantize_flag = "--quantize" if model_key in QUANTIZE_MODELS else ""
    cmd = (
        f"python train.py --model {model_key} {quantize_flag} "
        f"--epochs 5 --batch_size 8 --lr 2e-4"
    )
    print(f"\n{'='*60}")
    print(f"Egitim basliyor: {model_key}")
    print(f"Komut: {cmd}")
    print('='*60)

    result = subprocess.run(cmd, shell=True, capture_output=False)
    if result.returncode != 0:
        print(f"HATA: {model_key} egitimi basarisiz.")
    else:
        print(f"Tamamlandi: {model_key}")

print("\nTum modeller egitildi.")

## 7. Test Seti Degerlendirmesi & Karsilastirma

In [ ]:
# Test seti degerlendirme
model_dirs = [f"outputs/{k}" for k in SELECTED_MODELS if os.path.exists(f"outputs/{k}")]
print("Degerlendirilecek modeller:", model_dirs)

if model_dirs:
    dirs_arg = " ".join(model_dirs)
    !python evaluate.py --compare {dirs_arg} --out_dir outputs

In [ ]:
# Sonuclari yukle ve goster
import json

results_path = "outputs/comparison_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)

    import pandas as pd
    df = pd.DataFrame([{
        "Model"    : r["name"].split("/")[-1],
        "Accuracy" : r["accuracy"],
        "F1 Macro" : r["f1"],
        "Precision": r["precision"],
        "Recall"   : r["recall"],
        "ROC-AUC"  : r.get("roc_auc"),
    } for r in results])

    print("\n=== FINAL SONUCLARI ===")
    print(df.to_string(index=False))

    df.to_csv("outputs/final_results.csv", index=False)
    print("\nSonuclar kaydedildi: outputs/final_results.csv")

In [ ]:
# Gorsel karsilastirma grafigi
from IPython.display import Image
img_path = "outputs/model_comparison.png"
if os.path.exists(img_path):
    display(Image(img_path))

## 8. Hata Analizi (Error Analysis)

In [ ]:
# Hangi ornekler yanlis siniflandiriliyor?
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BEST_MODEL = "outputs/qwen"  # En iyi modeli buraya yazin

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    BEST_MODEL, device_map="auto", trust_remote_code=True
)
model.eval()

texts, labels = load_csv_texts("data/test.csv")
errors = []

for i, (text, label) in enumerate(zip(texts, labels)):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits
    pred = int(torch.argmax(logits, dim=-1))
    conf = float(torch.softmax(logits, dim=-1).max())

    if pred != label:
        errors.append({
            "idx"       : i,
            "true_label": label,
            "pred_label": pred,
            "confidence": round(conf, 4),
            "text_snippet": text[:200],
        })

print(f"Yanlis siniflandirilan: {len(errors)} / {len(texts)} ({len(errors)/len(texts)*100:.1f}%)")
print(f"\nIlk 5 hata:")
for err in errors[:5]:
    print(f"  [{err['idx']}] True={err['true_label']} Pred={err['pred_label']} Conf={err['confidence']}")
    print(f"       {err['text_snippet']}\n")

import json
with open("outputs/error_analysis.json", "w") as f:
    json.dump(errors, f, indent=2, ensure_ascii=False)
print("Hata analizi kaydedildi: outputs/error_analysis.json")

## 9. Sonuclari Drive'a Kaydet

In [ ]:
import shutil, os
from datetime import datetime

save_dir = f"/content/drive/MyDrive/third_party_breach_results_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(save_dir, exist_ok=True)

# Kaydet: sonuclar, grafikler, hata analizi (model agirliklarini degil -- cok buyuk)
for fname in ["outputs/final_results.csv", "outputs/comparison_results.json",
              "outputs/model_comparison.png", "outputs/error_analysis.json",
              "outputs/text_length_dist.png"]:
    if os.path.exists(fname):
        shutil.copy(fname, save_dir)

print(f"Sonuclar Drive'a kaydedildi: {save_dir}")

## Notlar

- Model agirliklarini kaydetmek icin her `outputs/<model>` dizinini Drive'a tasiyabilirsiniz.
- Rapor icin `final_results.csv` ve `model_comparison.png` dosyalari kullanilabilir.
- `error_analysis.json` dosyasi hata analizi bolumu icin referans alabilirsiniz.
- Zero-shot vs fine-tuned karsilastirmasi icin `zero_shot_results` degiskenini kullanin.